In [ ]:
import pandas as pd
!pip install emoji pyarabic nltk
import nltk
import emoji
import re
import pyarabic.araby as araby
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer
from nltk.tokenize import word_tokenize
nltk.download('punkt')        # ---------> 'punkt' is a pre-trained NLTK tokenizer model that enables correct splitting of text into words or sentences.
nltk.download('stopwords')    # ---------> 'stopwords' is an NLTK resource that provides lists of common words (like “the”, “and”) to filter out during text preprocessing.
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
!pip install gensim
!pip uninstall numpy -y #remove current numpy
!pip uninstall gensim -y
!pip install numpy==1.25.2
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 74.1 MB/s eta 0:00:00
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: gensim 4.4.0
Uninstalling gensim-4.4.0:
  Successfully uninstalled gensim-4.4.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 145.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
  Using cached gensim-4.4.0-cp312-cp312-manylinux_2_24_x86_64.many

In [ ]:
import gensim
from gensim.models import Word2Vec

In [ ]:
data = pd.read_csv("/content/AAFAQ_Dataset.csv")
data.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/AAFAQ_Dataset.csv'

In [ ]:
data.shape

(5009, 3)

In [ ]:
data.isnull().sum()

,0
QuestionText,0
Category,0
Answer,0


In [ ]:
data.duplicated().sum()

np.int64(188)

In [ ]:
data = data.drop_duplicates()

In [ ]:
def arabic_preprocessing(text):
    text = str(text)

    text = emoji.replace_emoji(text, replace=' ')

    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ؤ', 'و', text)
    text = re.sub(r'ئ', 'ي', text)
    text = re.sub(r'ء', '', text)

    #(tashkeel)
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)

    text = re.sub(r'ـ', '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
def classification_preprocessing(text):
    text = arabic_preprocessing(text)

    text = re.sub(r'[^ء-ي\s]', ' ', text) # non-Arabic characters
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# Classification
data['Question_for_classification'] = data['QuestionText'].apply(classification_preprocessing)
data['Answer_for_classification'] = data['Answer'].apply(classification_preprocessing)

# QA & Translation
data['Question_for_QA'] = data['QuestionText'].apply(arabic_preprocessing)
data['Answer_for_QA'] = data['Answer'].apply(arabic_preprocessing)

data['Question_for_translation'] = data['QuestionText'].apply(arabic_preprocessing)
data['Answer_for_translation'] = data['Answer'].apply(arabic_preprocessing)

In [ ]:
data['Question_tokens'] = data['Question_for_classification'].apply(lambda x: word_tokenize(str(x)))
data['Answer_tokens'] = data['Answer_for_classification'].apply(lambda x: word_tokenize(str(x)))

In [ ]:
question_words = ['من','ما','ماذا','متى','اين','كيف','لماذا','هل','كم','اي']

arabic_stopwords = stopwords.words('arabic')
arabic_stopwords = [w for w in arabic_stopwords if w not in question_words]

data['Question_tokens_no_stopwords'] = data['Question_tokens'].apply(lambda words: [word for word in words if word not in arabic_stopwords])
data['Answer_tokens_no_stopwords'] = data['Answer_tokens'].apply(lambda words: [word for word in words if word not in arabic_stopwords])

data[['Question_tokens', 'Question_tokens_no_stopwords']].head(5)

,Question_tokens,Question_tokens_no_stopwords
0,"[ايهما, افضل, الدراسة, في, السابق, ام, في, الو...","[ايهما, افضل, الدراسة, السابق, ام, الوقت, الحالي]"
1,"[اليس, القطن, عماد, الثروة, في, مصر]","[اليس, القطن, عماد, الثروة, مصر]"
2,"[اتصعد, الشمس, من, الشرق]","[اتصعد, الشمس, من, الشرق]"
3,"[اتعرف, البكتيريا, بانها, كاينات, حية, دقيقة]","[اتعرف, البكتيريا, بانها, كاينات, حية, دقيقة]"
4,"[ايتكون, الهوا, اساسا, من, النيتروجين]","[ايتكون, الهوا, اساسا, من, النيتروجين]"


In [ ]:
stemmer = ISRIStemmer()

data['Question_tokens_stemmed'] = data['Question_tokens_no_stopwords'].apply(lambda words: [stemmer.stem(word) for word in words])
data['Answer_tokens_stemmed'] = data['Answer_tokens_no_stopwords'].apply(lambda words: [stemmer.stem(word) for word in words])

data[['Question_tokens_no_stopwords', 'Question_tokens_stemmed']].head(5)

,Question_tokens_no_stopwords,Question_tokens_stemmed
0,"[ايهما, افضل, الدراسة, السابق, ام, الوقت, الحالي]","[ايه, فضل, درس, سبق, ام, وقت, الحالي]"
1,"[اليس, القطن, عماد, الثروة, مصر]","[الس, قطن, عمد, ثرة, مصر]"
2,"[اتصعد, الشمس, من, الشرق]","[صعد, شمس, من, شرق]"
3,"[اتعرف, البكتيريا, بانها, كاينات, حية, دقيقة]","[عرف, كتر, بان, كين, حية, دقق]"
4,"[ايتكون, الهوا, اساسا, من, النيتروجين]","[ايت, هوا, سسا, من, ترج]"


In [ ]:
data['Question_text_no_stopwords'] = data['Question_tokens_no_stopwords'].apply(lambda x: ' '.join(x))
data['Question_text_stemmed'] = data['Question_tokens_stemmed'].apply(lambda x: ' '.join(x))

data['Answer_text_no_stopwords'] = data['Answer_tokens_no_stopwords'].apply(lambda x: ' '.join(x))
data['Answer_text_stemmed'] = data['Answer_tokens_stemmed'].apply(lambda x: ' '.join(x))

In [ ]:
data.to_csv("processed_data.csv", index=False, encoding='utf-8-sig')

In [ ]:
data2 = pd.read_csv("/content/processed_data.csv")
data2.head()

,QuestionText,Category,Answer,Question_for_classification,Answer_for_classification,Question_for_QA,Answer_for_QA,Question_for_translation,Answer_for_translation,Question_tokens,Answer_tokens,Question_tokens_no_stopwords,Answer_tokens_no_stopwords,Question_tokens_stemmed,Answer_tokens_stemmed,Question_text_no_stopwords,Question_text_stemmed,Answer_text_no_stopwords,Answer_text_stemmed
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,التعليم,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,ايهما افضل الدراسة في السابق ام في الوقت الحالي,الدراسة في الوقت الحالي تعتبر افضل بسبب توفر ا...,ايهما افضل الدراسة في السابق ام في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر افضل بسبب توفر ا...,ايهما افضل الدراسة في السابق ام في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر افضل بسبب توفر ا...,"['ايهما', 'افضل', 'الدراسة', 'في', 'السابق', '...","['الدراسة', 'في', 'الوقت', 'الحالي', 'تعتبر', ...","['ايهما', 'افضل', 'الدراسة', 'السابق', 'ام', '...","['الدراسة', 'الوقت', 'الحالي', 'تعتبر', 'افضل'...","['ايه', 'فضل', 'درس', 'سبق', 'ام', 'وقت', 'الح...","['درس', 'وقت', 'الحالي', 'عبر', 'فضل', 'سبب', ...",ايهما افضل الدراسة السابق ام الوقت الحالي,ايه فضل درس سبق ام وقت الحالي,الدراسة الوقت الحالي تعتبر افضل بسبب توفر التك...,درس وقت الحالي عبر فضل سبب وفر كنولوج ورد علم حدث
1,أليس القطن عماد الثروة في مصر؟,الاقتصاد والعمل,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,اليس القطن عماد الثروة في مصر,القطن يعتبر من اهم المنتجات الزراعية في مصر وي...,اليس القطن عماد الثروة في مصر؟,القطن يعتبر من اهم المنتجات الزراعية في مصر، و...,اليس القطن عماد الثروة في مصر؟,القطن يعتبر من اهم المنتجات الزراعية في مصر، و...,"['اليس', 'القطن', 'عماد', 'الثروة', 'في', 'مصر']","['القطن', 'يعتبر', 'من', 'اهم', 'المنتجات', 'ا...","['اليس', 'القطن', 'عماد', 'الثروة', 'مصر']","['القطن', 'يعتبر', 'من', 'اهم', 'المنتجات', 'ا...","['الس', 'قطن', 'عمد', 'ثرة', 'مصر']","['قطن', 'عبر', 'من', 'اهم', 'نتج', 'زرع', 'مصر...",اليس القطن عماد الثروة مصر,الس قطن عمد ثرة مصر,القطن يعتبر من اهم المنتجات الزراعية مصر ويعد ...,قطن عبر من اهم نتج زرع مصر يعد من عمد ريس قصد صري
2,أتصعد الشمس من الشرق؟,التعليم,الشمس تصعد من الشرق.,اتصعد الشمس من الشرق,الشمس تصعد من الشرق,اتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.,اتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.,"['اتصعد', 'الشمس', 'من', 'الشرق']","['الشمس', 'تصعد', 'من', 'الشرق']","['اتصعد', 'الشمس', 'من', 'الشرق']","['الشمس', 'تصعد', 'من', 'الشرق']","['صعد', 'شمس', 'من', 'شرق']","['شمس', 'صعد', 'من', 'شرق']",اتصعد الشمس من الشرق,صعد شمس من شرق,الشمس تصعد من الشرق,شمس صعد من شرق
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,التعليم,البكتيريا تُعرف بأنها كائنات حية دقيقة.,اتعرف البكتيريا بانها كاينات حية دقيقة,البكتيريا تعرف بانها كاينات حية دقيقة,اتعرف البكتيريا بانها كاينات حية دقيقة؟,البكتيريا تعرف بانها كاينات حية دقيقة.,اتعرف البكتيريا بانها كاينات حية دقيقة؟,البكتيريا تعرف بانها كاينات حية دقيقة.,"['اتعرف', 'البكتيريا', 'بانها', 'كاينات', 'حية...","['البكتيريا', 'تعرف', 'بانها', 'كاينات', 'حية'...","['اتعرف', 'البكتيريا', 'بانها', 'كاينات', 'حية...","['البكتيريا', 'تعرف', 'بانها', 'كاينات', 'حية'...","['عرف', 'كتر', 'بان', 'كين', 'حية', 'دقق']","['كتر', 'عرف', 'بان', 'كين', 'حية', 'دقق']",اتعرف البكتيريا بانها كاينات حية دقيقة,عرف كتر بان كين حية دقق,البكتيريا تعرف بانها كاينات حية دقيقة,كتر عرف بان كين حية دقق
4,أيتكون الهواء أساساً من النيتروجين؟,التعليم,الهواء يتكون أساساً من النيتروجين.,ايتكون الهوا اساسا من النيتروجين,الهوا يتكون اساسا من النيتروجين,ايتكون الهوا اساسا من النيتروجين؟,الهوا يتكون اساسا من النيتروجين.,ايتكون الهوا اساسا من النيتروجين؟,الهوا يتكون اساسا من النيتروجين.,"['ايتكون', 'الهوا', 'اساسا', 'من', 'النيتروجين']","['الهوا', 'يتكون', 'اساسا', 'من', 'النيتروجين']","['ايتكون', 'الهوا', 'اساسا', 'من', 'النيتروجين']","['الهوا', 'يتكون', 'اساسا', 'من', 'النيتروجين']","['ايت', 'هوا', 'سسا', 'من', 'ترج']","['هوا', 'يتك', 'سسا', 'من', 'ترج']",ايتكون الهوا اساسا من النيتروجين,ايت هوا سسا من ترج,الهوا يتكون اساسا من النيتروجين,هوا يتك سسا من ترج


In [ ]:
!wget "https://bakrianoo.ewr1.vultrobjects.com/aravec/full_grams_sg_300_wiki.zip"

--2026-06-02 14:57:08--  https://bakrianoo.ewr1.vultrobjects.com/aravec/full_grams_sg_300_wiki.zip
Resolving bakrianoo.ewr1.vultrobjects.com (bakrianoo.ewr1.vultrobjects.com)... 108.61.0.122, 2001:19f0:0:22::100
Connecting to bakrianoo.ewr1.vultrobjects.com (bakrianoo.ewr1.vultrobjects.com)|108.61.0.122|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1488871452 (1.4G) [application/zip]
Saving to: ‘full_grams_sg_300_wiki.zip’

full_grams_sg_300_w 100%[===================>]   1.39G  15.4MB/s    in 2m 8s   

2026-06-02 14:59:17 (11.1 MB/s) - ‘full_grams_sg_300_wiki.zip’ saved [1488871452/1488871452]



In [ ]:
!unzip "full_grams_sg_300_wiki.zip"

Archive:  full_grams_sg_300_wiki.zip
  inflating: full_grams_sg_300_wiki.mdl  
  inflating: full_grams_sg_300_wiki.mdl.trainables.syn1neg.npy   bad CRC 025479a0  (should be 55675a21)
  inflating: full_grams_sg_300_wiki.mdl.wv.vectors.npy  


In [ ]:
sg_pretrained = gensim.models.Word2Vec.load("/content/full_grams_sg_300_wiki.mdl")

In [ ]:
def get_sentence_vector(words, model):
    vectors = []

    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [ ]:
import numpy as np
X_sg_pretrained_no_stop = np.array(data2['Question_tokens_no_stopwords'].apply(lambda x: get_sentence_vector(x, sg_pretrained)).tolist())
X_sg_pretrained_no_stop.shape

(4821, 300)

In [ ]:
X_sg_pretrained_stemmed = np.array(data2['Question_tokens_stemmed'].apply(lambda x: get_sentence_vector(x, sg_pretrained)).tolist())
X_sg_pretrained_stemmed.shape

(4821, 300)

In [ ]:
!wget "https://bakrianoo.ewr1.vultrobjects.com/aravec/full_grams_cbow_300_wiki.zip"

--2026-06-02 15:12:37--  https://bakrianoo.ewr1.vultrobjects.com/aravec/full_grams_cbow_300_wiki.zip
Resolving bakrianoo.ewr1.vultrobjects.com (bakrianoo.ewr1.vultrobjects.com)... 108.61.0.122, 2001:19f0:0:22::100
Connecting to bakrianoo.ewr1.vultrobjects.com (bakrianoo.ewr1.vultrobjects.com)|108.61.0.122|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1491895880 (1.4G) [application/zip]
Saving to: ‘full_grams_cbow_300_wiki.zip’

full_grams_cbow_300 100%[===================>]   1.39G  15.4MB/s    in 96s     

2026-06-02 15:14:15 (14.8 MB/s) - ‘full_grams_cbow_300_wiki.zip’ saved [1491895880/1491895880]



In [ ]:
!unzip "full_grams_cbow_300_wiki"

Archive:  full_grams_cbow_300_wiki.zip
  inflating: full_grams_cbow_300_wiki.mdl  
  inflating: full_grams_cbow_300_wiki.mdl.trainables.syn1neg.npy  
  inflating: full_grams_cbow_300_wiki.mdl.wv.vectors.npy  


In [ ]:
cbow_pretrained = gensim.models.Word2Vec.load("/content/full_grams_cbow_300_wiki.mdl")

In [ ]:
X_cbow_pretrained_no_stop = np.array(data2['Question_tokens_no_stopwords'].apply(lambda x: get_sentence_vector(x, cbow_pretrained)).tolist())
X_cbow_pretrained_no_stop.shape

(4821, 300)

In [ ]:
X_cbow_pretrained_stemmed = np.array(data2['Question_tokens_no_stopwords'].apply(lambda x: get_sentence_vector(x, cbow_pretrained)).tolist())
X_cbow_pretrained_stemmed.shape

(4821, 300)

In [ ]:
np.save("sg_no_stop_embeddings.npy", X_sg_pretrained_no_stop)
np.save("sg_stemmed_embeddings.npy", X_sg_pretrained_stemmed)
np.save("cbow_no_stop_embeddings.npy", X_cbow_pretrained_no_stop)
np.save("cbow_stemmed_embeddings.npy", X_cbow_pretrained_stemmed)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('aubmindlab/bert-base-arabertv02')
model = AutoModel.from_pretrained('aubmindlab/bert-base-arabertv02')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def get_bert_embeddings(texts):
    inputs = tokenizer(
        texts.tolist(),
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    embeddings = outputs.last_hidden_state.mean(dim=1)

    return embeddings.cpu().numpy()

In [ ]:
X_bert = get_bert_embeddings(data2['Question_for_classification'])
X_bert.shape

(4821, 768)

In [ ]:
np.save("bert_embeddings.npy", X_bert)

In [ ]:
model_name = "Qwen/Qwen3-Embedding-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name,trust_remote_code=True)
model = AutoModel.from_pretrained(model_name,trust_remote_code=True)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [ ]:
def get_qwen_embeddings(texts):

    inputs = tokenizer(texts.tolist(),return_tensors='pt',padding=True,truncation=True,max_length=128)

    with torch.no_grad():
        outputs = model(**inputs)

    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.float().cpu().numpy()

In [ ]:
X_qwen = get_qwen_embeddings(data2['Question_for_classification'])
X_qwen.shape

(4821, 1024)

In [ ]:
np.save("qwen_embeddings.npy", X_qwen)

In [ ]:
model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def get_e5_embeddings(texts):

    texts = [f"passage: {t}" for t in texts.tolist()]

    inputs = tokenizer(
        texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    embeddings = outputs.last_hidden_state.mean(dim=1)

    return embeddings.cpu().numpy()

In [ ]:
X_e5 = get_e5_embeddings(data2['Question_for_classification'])
X_e5.shape

(4821, 1024)

In [ ]:
np.save("e5_embeddings.npy", X_e5)

In [ ]:
model_name = "BAAI/bge-m3"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
def get_bge_embeddings(texts):
    inputs = tokenizer(
        texts.tolist(),
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    embeddings = outputs.last_hidden_state.mean(dim=1)

    return embeddings.cpu().numpy()

In [ ]:
X_bge = get_bge_embeddings(data2['Question_for_classification'])
X_bge.shape

(4821, 1024)

In [ ]:
np.save("bge_embeddings.npy", X_bge)

In [ ]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 8.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.4-py3-none-any.whl (314 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653905 sha256=71ff477f171c07fb5ed39c05990e7b8e177d0bcbf71e4b5c55ab1320cdea65b4
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [ ]:
import fasttext

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz

--2026-06-02 19:56:00--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ar.300.bin.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.168.4, 65.9.168.62, 65.9.168.81, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.168.4|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4500982519 (4.2G) [application/octet-stream]
Saving to: ‘cc.ar.300.bin.gz’

cc.ar.300.bin.gz    100%[===================>]   4.19G  23.4MB/s    in 3m 7s   

2026-06-02 19:59:08 (22.9 MB/s) - ‘cc.ar.300.bin.gz’ saved [4500982519/4500982519]



In [ ]:
!gunzip cc.ar.300.bin.gz

In [ ]:
ft_model = fasttext.load_model("/content/cc.ar.300.bin")

In [ ]:
def get_fasttext_vector(words):
    vectors = []

    for word in words:
        vectors.append(ft_model.get_word_vector(word))

    return np.mean(vectors, axis=0)

In [ ]:
X_fasttext = np.array(data2['Question_tokens_no_stopwords'].apply(lambda x: get_fasttext_vector(x)).tolist())
X_fasttext.shape

(4821, 300)

In [ ]:
np.save("fasttext_no_stop_embeddings.npy",X_fasttext)

In [ ]:
X_fasttext_stemmed = np.array(data2['Question_tokens_stemmed'].apply(lambda x: get_fasttext_vector(x)).tolist())
X_fasttext_stemmed.shape

(4821, 300)

In [ ]:
np.save("fasttext_stemmed_embeddings.npy",X_fasttext_stemmed)

In [ ]:
import ast
data2['Question_tokens'] = data2['Question_tokens'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [ ]:
sg_trained = Word2Vec(data2['Question_tokens'],vector_size=300,min_count=1,window=3,sg=1)

In [ ]:
cbow_trained = Word2Vec(data2['Question_tokens'],vector_size=300,min_count=1,window=3,sg=0)

In [ ]:
def get_sentence_vector(words, model):
    vectors = []

    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [ ]:
import numpy as np
X_sg_trained = np.array(data2['Question_tokens'].apply(lambda x: get_sentence_vector(x, sg_trained)).tolist())
X_sg_trained.shape

(4821, 300)

In [ ]:
np.save("sg_trained_embeddings.npy", X_sg_trained)

In [ ]:
X_cbow_trained = np.array(data2['Question_tokens'].apply(lambda x: get_sentence_vector(x, cbow_trained)).tolist())
X_cbow_trained.shape

(4821, 300)

In [ ]:
np.save("cbow_trained_embeddings.npy", X_cbow_trained)